# Production Multi-*m* Debris Spectrum Analysis — 4th-Order D3
## Lax Centennial Paper: "What Remains When Integrability Fails?"

Upgraded solver: D3 uses 4th-order 7-point central stencil (was 2nd-order 5-point).

**Improvement at N=512 (m=3 benchmark):**

| Metric | D3 2nd-order | D3 4th-order | Change |
|---|---|---|---|
| noise_L2 | 2.24e-02 | 6.50e-03 | **3.5× lower** |
| SNR | 2.2 | **16.7** | **7.6× higher** |
| Amp preserved | 0.9989 | 1.0001 | near-exact |

**Runtime estimates (4 cases, with noise controls):**

| N | Total time |
|---|------------|
| 512 | ~5 min |
| 1024 | ~50 min |
| 2048 | ~12 hr |

## ① Configuration

In [ ]:
N         = 1024       # 512 (validation), 1024 (production), 2048 (high-res)
L         = 100.0
q         = 3
delta     = 0.5
A1        = 1.0
A2        = 0.5
x1_init   = 20.0
x2_init   = 40.0
cfl_safety = 0.8
m_values  = [2, 3, 4, 5]

run_noise_controls = True
output_dir = '/content'

cfg = {
    'N': N, 'L': L, 'q': q, 'delta': delta,
    'A1': A1, 'A2': A2,
    'x1_init': x1_init, 'x2_init': x2_init,
    'cfl_safety': cfl_safety,
    'sep_margin': 5.0, 'sep_buffer': 10.0, 'min_t_extra': 40.0,
    'm_values': m_values,
    'output_dir': output_dir,
}

import os
os.makedirs(output_dir, exist_ok=True)
dx = L / N
dt_est = cfl_safety * 2.83 * dx**3 / (4.0 * max(m_values))
print(f"Grid: N={N}, dx={dx:.6f}, dt≈{dt_est:.4e}")
print(f"D3 stencil: 4th-order (7-point)")
print(f"Output → {os.path.abspath(output_dir)}")

## ② Imports

In [ ]:
import numpy as np
from scipy.fft import fft, ifft, fftfreq
from scipy.signal import find_peaks
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import time
import json
import os

## ③ Compacton Physics

In [ ]:
def compacton_profile(m, A, x, x0=0.0):
    alpha = 2.0 / (m - 1.0)
    ell = 2.0 * m / (m - 1.0)
    hw = np.pi * ell / 2.0
    xi = x - x0
    u = np.zeros_like(x, dtype=np.float64)
    inside = np.abs(xi) < hw
    u[inside] = A * np.cos(xi[inside] / ell) ** alpha
    return u

def compacton_speed(m, A):
    return (m + 1.0) / (2.0 * m) * A ** (m - 1.0)

def compacton_half_width(m):
    ell = 2.0 * m / (m - 1.0)
    return np.pi * ell / 2.0

def compute_separation_time(m, A1, A2, x1i, x2i, margin=5.0, buffer=10.0, min_extra=40.0):
    c1 = compacton_speed(m, A1); c2 = compacton_speed(m, A2)
    hw = compacton_half_width(m)
    if c1 <= c2:
        return 100.0, min_extra, {'warning': 'no collision'}
    delta_c = c1 - c2; t_col = (x2i - x1i) / delta_c
    sep_needed = 2 * hw + margin; t_sep = sep_needed / delta_c
    t_extra = max(t_sep + buffer, min_extra)
    return t_col, t_extra, {
        'c1': c1, 'c2': c2, 'delta_c': delta_c, 't_col': t_col,
        'hw': hw, 'sep_needed': sep_needed, 't_sep': t_sep,
        't_extra': t_extra, 't_end': t_col + t_extra,
    }

## ④ FD Solver — 4th-Order D3

In [ ]:
def solve_rk4_fd(m, u0, x, t_end, q, delta=0.5, safety=0.8, progress_interval=30):
    """FD solver with 4th-order D1 and D3, 2nd-order D2.

    Key change from prior version: D3 upgraded from 2nd-order to 4th-order
    7-point central stencil. This pushes the FD/exact dispersion divergence
    to higher k, dramatically reducing noise for m>=3 and providing a
    cleaner baseline for m=2 diagnostics.

    RHS(u) = -LP[D1(u^m)] - D3(u^m) + HP[delta * max|u|^{m-1} * dx * D2(u)]
    """
    N = len(x); dx = x[1] - x[0]
    n_keep = N // (2 * q)
    lp = np.zeros(N); lp[0] = 1.0
    lp[1:n_keep] = 1.0; lp[N - n_keep + 1:] = 1.0
    hp = 1.0 - lp

    def D1(u):  # 4th-order central
        return (-np.roll(u,-2) + 8*np.roll(u,-1)
                - 8*np.roll(u,1) + np.roll(u,2)) / (12*dx)
    def D2(u):  # 2nd-order central (HP dissipation only)
        return (np.roll(u,-1) - 2*u + np.roll(u,1)) / (dx**2)
    def D3(u):  # 4th-order central, 7-point stencil (UNFILTERED)
        return (-np.roll(u,-3) + 8*np.roll(u,-2) - 13*np.roll(u,-1)
                + 13*np.roll(u,1) - 8*np.roll(u,2) + np.roll(u,3)) / (8*dx**3)
    def filt(u, mask):
        return np.real(ifft(fft(u) * mask))

    def rhs(u):
        um = np.sign(u) * np.abs(u)**m
        umax_m1 = max(np.max(np.abs(u))**(m-1), 1e-10)
        return (-filt(D1(um), lp) - D3(um)
                + filt(delta * umax_m1 * dx * D2(u), hp))

    dt = safety * 2.83 * dx**3 / (4.0 * m)
    u = u0.copy(); t = 0.0; nsteps = 0
    t0 = time.time(); last_report = t0
    total_steps_est = int(np.ceil(t_end / dt))

    while t < t_end - 1e-14:
        dt_use = min(dt, t_end - t)
        k1 = rhs(u); k2 = rhs(u + 0.5*dt_use*k1)
        k3 = rhs(u + 0.5*dt_use*k2); k4 = rhs(u + dt_use*k3)
        u += (dt_use/6.0)*(k1 + 2*k2 + 2*k3 + k4)
        t += dt_use; nsteps += 1

        if progress_interval > 0:
            now = time.time()
            if now - last_report > progress_interval:
                pct = 100.0 * t / t_end
                elapsed = now - t0
                eta = elapsed / max(pct, 0.1) * (100 - pct)
                print(f"    Step {nsteps:,}/{total_steps_est:,} "
                      f"({pct:.1f}%), elapsed {elapsed:.0f}s, ETA {eta:.0f}s",
                      flush=True)
                last_report = now

    return u, time.time() - t0, nsteps

## ⑤ Analysis Functions

In [ ]:
def extract_debris(u_post, m, x):
    dx = x[1] - x[0]
    peaks, _ = find_peaks(u_post, height=0.02, distance=10)
    if len(peaks) < 2:
        print("    WARNING: < 2 peaks found")
        return u_post, 0, 0, 0, 0, np.inf
    heights = u_post[peaks]; idx_sort = np.argsort(heights)[::-1]
    p1, p2 = peaks[idx_sort[0]], peaks[idx_sort[1]]
    def residual(params):
        a1, x01, a2, x02 = params
        if a1 < 0.01 or a2 < 0.01: return 1e10
        fit = compacton_profile(m, a1, x, x01) + compacton_profile(m, a2, x, x02)
        return np.sum((u_post - fit)**2) * dx
    res = minimize(residual, [u_post[p1], x[p1], u_post[p2], x[p2]],
                   method='Nelder-Mead',
                   options={'xatol': 1e-10, 'fatol': 1e-16, 'maxiter': 10000})
    A1f, x1f, A2f, x2f = res.x
    fitted = compacton_profile(m, A1f, x, x1f) + compacton_profile(m, A2f, x, x2f)
    return u_post - fitted, A1f, x1f, A2f, x2f, res.fun

def extract_single_noise(u_post, m, x):
    dx = x[1] - x[0]; pk = np.argmax(u_post)
    res = minimize(
        lambda p: np.sum((u_post - compacton_profile(m, p[0], x, p[1]))**2) * dx,
        [u_post[pk], x[pk]], method='Nelder-Mead',
        options={'xatol': 1e-10, 'fatol': 1e-16, 'maxiter': 10000})
    return u_post - compacton_profile(m, res.x[0], x, res.x[1]), res.x[0], res.x[1]

def get_spectrum(signal, x):
    N = len(x); dx = x[1] - x[0]; s_hat = fft(signal)
    kwave = fftfreq(N, dx) * 2 * np.pi; pos = kwave >= 0
    kp, sp = kwave[pos], np.abs(s_hat[pos]); si = np.argsort(kp)
    return kp[si], sp[si]

def analyze_below_cutoff(k, spec, k_cut_phys, k_target=1.0):
    below = k < k_cut_phys; k_b = k[below]; s_b = spec[below]
    if len(k_b) == 0:
        return {'w_k1':0,'k_peak':0,'s_peak':0,'notch_ratio':999,
                'spec_L2':0,'k_below':k_b,'spec_below':s_b}
    k1_idx = np.argmin(np.abs(k_b - k_target)); w_k1 = s_b[k1_idx]
    valid = k_b > 0.1
    if np.any(valid):
        pi = np.argmax(s_b[valid]); k_peak = k_b[valid][pi]; s_peak = s_b[valid][pi]
    else:
        k_peak, s_peak = 0.0, 0.0
    notch = w_k1 / s_peak if s_peak > 0 else 999.0
    dk = k_b[1] - k_b[0] if len(k_b) > 1 else 1.0
    spec_L2 = np.sqrt(np.sum(s_b**2) * dk)
    k_near = (k_b > 0.5) & (k_b < 2.0)
    if np.any(k_near):
        lmin_idx = np.argmin(s_b[k_near])
        k_lmin, v_lmin = k_b[k_near][lmin_idx], s_b[k_near][lmin_idx]
    else:
        k_lmin, v_lmin = 1.0, w_k1
    return {'w_k1': w_k1, 'k_peak': k_peak, 's_peak': s_peak,
            'notch_ratio': notch, 'spec_L2': spec_L2,
            'k_lmin': k_lmin, 'v_lmin': v_lmin,
            'k_below': k_b, 'spec_below': s_b}

def compute_cutoff(N, q, L):
    N_modes = N // 2
    k_cut_idx = int(np.ceil(N_modes / q))
    return k_cut_idx, 2 * np.pi * k_cut_idx / L

## ⑥ Single-Case Runner

In [ ]:
def run_single_case(m, cfg, run_noise=True):
    N = cfg['N']; L = cfg['L']; q = cfg['q']
    A1 = cfg['A1']; A2 = cfg['A2']
    x1i = cfg['x1_init']; x2i = cfg['x2_init']

    x = np.linspace(0, L, N, endpoint=False); dx = x[1] - x[0]
    alpha = 2.0 / (m - 1.0)
    k_cut_idx, k_cut_phys = compute_cutoff(N, q, L)

    t_col, t_extra, sep_info = compute_separation_time(
        m, A1, A2, x1i, x2i,
        margin=cfg['sep_margin'], buffer=cfg['sep_buffer'],
        min_extra=cfg['min_t_extra'])
    t_end = t_col + t_extra

    print(f"\n{'='*70}")
    print(f"K({m},{m}): α={alpha:.3f}, N={N}, dx={dx:.6f}")
    print(f"  c1={sep_info['c1']:.6f}, c2={sep_info['c2']:.6f}, Δc={sep_info['delta_c']:.6f}")
    print(f"  t_col={t_col:.1f}, t_extra={t_extra:.1f}, t_end={t_end:.1f}")
    print(f"  hw={sep_info['hw']:.2f}, κ_cut={k_cut_phys:.3f}")

    u0 = compacton_profile(m, A1, x, x1i) + compacton_profile(m, A2, x, x2i)
    mass_init = np.sum(u0) * dx; energy_init = np.sum(u0**2) * dx
    progress = 30 if N >= 1024 else 0

    print(f"  Running collision...", flush=True)
    u_col, t_elapsed, nsteps = solve_rk4_fd(
        m, u0, x, t_end, q, cfg['delta'], cfg['cfl_safety'],
        progress_interval=progress)
    mass_err = abs(np.sum(u_col)*dx - mass_init) / abs(mass_init)
    print(f"  Done: {t_elapsed:.1f}s, {nsteps:,} steps, mass_err={mass_err:.2e}")

    noise_data = None
    if run_noise:
        u0_sin = compacton_profile(m, A1, x, x1i)
        print(f"  Running noise control...", flush=True)
        u_sin, t_sin, _ = solve_rk4_fd(m, u0_sin, x, t_end, q,
                                         cfg['delta'], cfg['cfl_safety'],
                                         progress_interval=progress)
        noise, Anf, _ = extract_single_noise(u_sin, m, x)
        k_n, s_n = get_spectrum(noise, x)
        na = analyze_below_cutoff(k_n, s_n, k_cut_phys)
        noise_L2 = np.sqrt(np.sum(noise**2) * dx)
        print(f"  Noise: L2={noise_L2:.4e}, Amp={Anf:.6f}")
        noise_data = {'noise': noise, 'k_n': k_n, 's_n': s_n,
                      'na': na, 'noise_L2': noise_L2, 'A_fitted': Anf}

    debris, A1f, x1f, A2f, x2f, fit_res = extract_debris(u_col, m, x)
    debris_L2 = np.sqrt(np.sum(debris**2) * dx)
    energy_frac = np.sum(debris**2) * dx / max(energy_init, 1e-30)

    peaks, _ = find_peaks(u_col, height=0.02, distance=10); peak_sep = 0.0
    if len(peaks) >= 2:
        heights = u_col[peaks]; idx_sort = np.argsort(heights)[::-1]
        p1, p2 = peaks[idx_sort[0]], peaks[idx_sort[1]]
        peak_sep = min(abs(x[p1]-x[p2]), L - abs(x[p1]-x[p2]))
    hw = compacton_half_width(m)
    sep_ok = "✓" if peak_sep > 2*hw else "✗ OVERLAP"
    print(f"  Fitted: A1={A1f:.6f}, A2={A2f:.6f}")
    print(f"  Separation: {peak_sep:.2f} vs 2hw={2*hw:.2f}  {sep_ok}")
    print(f"  Debris L2={debris_L2:.4e}, E_deb/E_tot={energy_frac:.4e}")

    k_d, s_d = get_spectrum(debris, x)
    da = analyze_below_cutoff(k_d, s_d, k_cut_phys)
    snr_below = (da['spec_L2'] / max(noise_data['na']['spec_L2'], 1e-30)
                 if noise_data else None)
    snr_full = (debris_L2 / max(noise_data['noise_L2'], 1e-30)
                if noise_data else None)
    print(f"  Notch={da['notch_ratio']:.4f}, |ŵ(1)|={da['w_k1']:.4e}, "
          f"peak={da['s_peak']:.4e}@k={da['k_peak']:.2f}")
    if snr_below: print(f"  SNR: full={snr_full:.1f}, below-cut={snr_below:.1f}")

    return {
        'm': m, 'alpha': alpha, 'N': N, 'dx': dx, 'L': L, 'q': q,
        'A1': A1, 'A2': A2, 'x': x, 'u_col': u_col, 'debris': debris,
        'k_d': k_d, 's_d': s_d, 'da': da,
        'k_cut_idx': k_cut_idx, 'k_cut_phys': k_cut_phys,
        'A1_fitted': A1f, 'A2_fitted': A2f,
        'x1_fitted': x1f, 'x2_fitted': x2f,
        'debris_L2': debris_L2, 'energy_frac': energy_frac,
        'mass_err': mass_err, 'peak_sep': peak_sep,
        'fit_residual': fit_res, 'sep_info': sep_info,
        't_col': t_col, 't_extra': t_extra, 't_end': t_end,
        'elapsed': t_elapsed, 'nsteps': nsteps,
        'snr_below': snr_below, 'snr_full': snr_full,
        'noise_data': noise_data,
    }

## ⑦ Run All Cases

In [ ]:
results = {}
total_t0 = time.time()
for m_val in cfg['m_values']:
    results[m_val] = run_single_case(m_val, cfg, run_noise=run_noise_controls)
total_elapsed = time.time() - total_t0
print(f"\n{'='*70}")
print(f"ALL DONE: {total_elapsed:.1f}s ({total_elapsed/60:.1f} min)")

## ⑧ Summary Table

In [ ]:
print(f"\n{'='*100}")
print(f"RESULTS: N={cfg['N']}, D3=4th-order, q={cfg['q']}")
print(f"{'='*100}")
header = (f"{'m':>3} {'α':>6} {'t_ext':>6} {'sep':>6} {'A1f':>8} {'A2f':>8} "
          f"{'deb_L2':>10} {'E_deb':>10} {'notch':>7} {'|ŵ(1)|':>10} "
          f"{'peak':>10} {'k_pk':>5} {'SNR':>6} {'time':>7}")
print(header); print('-' * 100)
for m_val in sorted(results.keys()):
    r = results[m_val]
    snr_str = f'{r["snr_below"]:.1f}' if r['snr_below'] else '  N/A'
    print(f"{m_val:>3} {r['alpha']:>6.3f} {r['t_extra']:>6.1f} "
          f"{r['peak_sep']:>6.2f} "
          f"{r['A1_fitted']:>8.5f} {r['A2_fitted']:>8.5f} "
          f"{r['debris_L2']:>10.3e} {r['energy_frac']:>10.3e} "
          f"{r['da']['notch_ratio']:>7.4f} "
          f"{r['da']['w_k1']:>10.3e} {r['da']['s_peak']:>10.3e} "
          f"{r['da']['k_peak']:>5.2f} "
          f"{snr_str:>6} {r['elapsed']:>6.1f}s")
print('-' * 100)
print(f"\nSeparation check:")
for m_val in sorted(results.keys()):
    r = results[m_val]; hw = r['sep_info']['hw']
    ok = "✓" if r['peak_sep'] > 2*hw else "✗ OVERLAPPING"
    print(f"  m={m_val}: peak_sep={r['peak_sep']:.2f}, 2hw={2*hw:.2f}  {ok}")

## ⑨ 4-Panel Spectrum Figure

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
axes_flat = axes.flatten()
labels = ['(a)', '(b)', '(c)', '(d)']
colors = ['#1f77b4', '#2ca02c', '#d62728', '#9467bd']
for idx, m_val in enumerate(cfg['m_values']):
    if m_val not in results: continue
    ax = axes_flat[idx]; r = results[m_val]; da = r['da']
    k_b, s_b = da['k_below'], da['spec_below']; valid = k_b > 0.05
    ax.semilogy(k_b[valid], s_b[valid]+1e-16, '-', color=colors[idx], lw=0.8)
    ax.axvline(1.0, color='red', ls='--', lw=1.0, alpha=0.7)
    k1_idx = np.argmin(np.abs(k_b - 1.0))
    ax.plot(k_b[k1_idx], s_b[k1_idx]+1e-16, 'rv', ms=8, zorder=10,
            label=f'$k=1$: {da["w_k1"]:.1e}')
    ax.plot(da['k_peak'], da['s_peak'], 'k^', ms=7, zorder=10,
            label=f'peak: {da["s_peak"]:.1e} @ k={da["k_peak"]:.2f}')
    nd = r.get('noise_data')
    if nd:
        na = nd['na']; k_nb = na['k_below']; s_nb = na['spec_below']
        vn = k_nb > 0.05
        ax.semilogy(k_nb[vn], s_nb[vn]+1e-16, 'k--', lw=0.5, alpha=0.4, label='noise floor')
    snr_str = f'SNR = {r["snr_below"]:.1f}' if r['snr_below'] else ''
    ax.text(0.97, 0.60,
            f'notch = {da["notch_ratio"]:.3f}\nlocal min: k={da["k_lmin"]:.2f}\n'
            f'$E_{{\\mathrm{{deb}}}}/E_{{\\mathrm{{tot}}}} = {r["energy_frac"]:.1e}$\n{snr_str}',
            transform=ax.transAxes, fontsize=8, va='top', ha='right',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='wheat', alpha=0.8))
    ax.set_xlabel('$k$'); ax.set_ylabel('$|\hat{w}(k)|$')
    ax.set_title(f'{labels[idx]} K({m_val},{m_val}), $\\alpha = {r["alpha"]:.2g}$')
    ax.legend(fontsize=7, loc='upper right'); ax.grid(True, alpha=0.15)
fig.suptitle(f'Debris Spectra: Spectral Notch at $k=1$ — D3 4th-order, N={cfg["N"]}',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(os.path.join(cfg['output_dir'], f'figure2_D3v4_N{cfg["N"]}.png'), dpi=150, bbox_inches='tight')
fig.savefig(os.path.join(cfg['output_dir'], f'figure2_D3v4_N{cfg["N"]}.pdf'), dpi=300, bbox_inches='tight')
plt.show()

## ⑩ Save JSON

In [ ]:
data = {
    'config': {k: cfg[k] for k in ['N','L','q','delta','A1','A2','cfl_safety']},
    'config_notes': 'D3=4th-order 7-point stencil',
    'cutoff': dict(zip(['k_cut_idx','k_cut_phys'],
                       [int(compute_cutoff(cfg['N'],cfg['q'],cfg['L'])[0]),
                        float(compute_cutoff(cfg['N'],cfg['q'],cfg['L'])[1])])),
    'results': {},
}
for m_val in sorted(results.keys()):
    r = results[m_val]
    entry = {
        'alpha': r['alpha'], 'dx': float(r['dx']),
        'c1': float(r['sep_info']['c1']), 'c2': float(r['sep_info']['c2']),
        'delta_c': float(r['sep_info']['delta_c']),
        't_col': float(r['t_col']), 't_extra': float(r['t_extra']),
        't_end': float(r['t_end']), 'hw': float(r['sep_info']['hw']),
        'peak_sep': float(r['peak_sep']),
        'A1_fitted': float(r['A1_fitted']), 'A2_fitted': float(r['A2_fitted']),
        'fit_residual': float(r['fit_residual']),
        'debris_L2': float(r['debris_L2']), 'energy_frac': float(r['energy_frac']),
        'mass_err': float(r['mass_err']),
        'notch_ratio': float(r['da']['notch_ratio']),
        'w_k1': float(r['da']['w_k1']),
        'k_peak': float(r['da']['k_peak']), 's_peak': float(r['da']['s_peak']),
        'k_lmin': float(r['da']['k_lmin']), 'v_lmin': float(r['da']['v_lmin']),
        'spec_L2_below': float(r['da']['spec_L2']),
        'snr_below': float(r['snr_below']) if r['snr_below'] else None,
        'snr_full': float(r['snr_full']) if r['snr_full'] else None,
        'elapsed_s': float(r['elapsed']), 'nsteps': r['nsteps'],
    }
    if r['noise_data']:
        entry['noise_L2'] = float(r['noise_data']['noise_L2'])
        entry['noise_specL2_below'] = float(r['noise_data']['na']['spec_L2'])
    data['results'][str(m_val)] = entry

json_path = os.path.join(cfg['output_dir'], f'measurements_D3v4_N{cfg["N"]}.json')
with open(json_path, 'w') as f:
    json.dump(data, f, indent=2)
print(f"Saved {json_path}")

# Uncomment for Colab download:
# from google.colab import files
# files.download(json_path)
# files.download(os.path.join(cfg['output_dir'], f'figure2_D3v4_N{cfg["N"]}.pdf'))